# **Análise Bivariada da Inflação**

Este notebook foi construído para identificar as variáveis mais associadas à variável alvo `inflation_target`, usando a **correlação de Pearson** como critério de seleção de uma *shorlist*.

A análise é organizada em quatro blocos:

- **correlação "contemporânea" em nível**, que define diretamente a shortlist;
- **diagnóstico da forma funcional**, para perceber que tipo de relação melhor ajusta cada variável à inflação;
- **correlação com lags das variáveis explicativas**, mantendo a inflação no presente;
- **autocorrelação da própria inflação**, para inspecionar a persistência temporal da variável alvo.

## **Critério metodológico adotado**

A shortlist final é construída a partir do ranking de `|Pearson|` em nível. Isto significa que:

- a análise de lags não entra como critério de seleção;
- o diagnóstico de forma funcional não entra como critério de seleção.

Assim, os lags e o `R²` funcional são usados apenas como **informação complementar** para interpretar melhor as variáveis selecionadas.


In [ ]:
# Imports
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Qualidade visual
%config InlineBackend.figure_format = 'svg'
sns.set_theme(style='whitegrid')

primary_color = '#5B8CC0'
secondary_color = '#F28E2B'

: 

## **Carregamento e preparação dos dados**

Por defeito, o notebook utiliza `dados/CompactedData.csv` que tem os dados raw.


In [ ]:
target = 'inflation_target'  # Variável alvo a analisar
top_n = 10                   # Número de variáveis a manter na shortlist final
max_lag = 12                 # Lag máximo a testar para as variáveis explicativas e para a inflação
exclude_vars = []            # Lista opcional de variáveis a excluir da análise

primary_path = Path('dados/CompactedData.csv')
fallback_path = Path('dados/train_complete.csv')

data_path = primary_path if primary_path.exists() else fallback_path
if not data_path.exists():
    raise FileNotFoundError('Não foi encontrado nem df_hybrid.csv nem dados/CompactedData.csv.')

df = pd.read_csv(data_path)

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date').sort_index()

# Converter tudo para numérico e eliminar colunas completamente vazias
df = df.apply(pd.to_numeric, errors='coerce')
df = df.dropna(axis=1, how='all')

candidate_cols = [
    col for col in df.columns
    if col != target and col not in exclude_vars
]

analysis_df = df[[target] + candidate_cols].copy()

print(f'Ficheiro utilizado: {data_path}')
print(f'Observações: {analysis_df.shape[0]}')
print(f'Variáveis candidatas: {len(candidate_cols)}')
display(analysis_df[[target]].describe().T)

## **Funções auxiliares**

As funções abaixo separam claramente:

- o ranking principal por `Pearson` em nível;
- o diagnóstico da forma funcional;
- a análise de lags das variáveis explicativas;
- a autocorrelação da inflação.


In [ ]:
def align_pair(frame, x_col, y_col):
    """
    Alinha duas séries usando apenas observações simultaneamente disponíveis.

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame com as séries temporais.
    x_col : str
        Nome da primeira série.
    y_col : str
        Nome da segunda série.

    Returns
    -------
    pair : pd.DataFrame
        DataFrame com duas colunas, `x` e `y`, contendo apenas observações
        em que ambas as séries estão disponíveis.
    """
    pair = frame[[x_col, y_col]].dropna().copy()
    pair.columns = ['x', 'y']
    return pair



def compute_pairwise_stats(frame, target):
    """
    Calcula a correlação de Pearson em nível, no mesmo período, entre cada variável
    explicativa e a variável alvo.

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame com a variável alvo e as variáveis explicativas.
    target : str
        Nome da variável alvo.

    Returns
    -------
    out : pd.DataFrame
        Tabela com uma linha por variável explicativa e as colunas:
        - `variable`: nome da variável;
        - `n_obs`: número de observações usadas no cálculo;
        - `pearson`: correlação de Pearson em nível;
        - `abs_pearson`: valor absoluto da correlação;
        - `rank_pearson_level`: posição da variável no ranking de `abs_pearson`.
    """
    rows = []

    for col in frame.columns:
        if col == target:
            continue

        pair = align_pair(frame, col, target)
        if len(pair) == 0:
            continue

        pearson = pair['x'].corr(pair['y'], method='pearson')

        rows.append({
            'variable': col,
            'n_obs': len(pair),
            'pearson': pearson,
            'abs_pearson': abs(pearson),
        })

    out = pd.DataFrame(rows).sort_values('abs_pearson', ascending=False).reset_index(drop=True)
    out['rank_pearson_level'] = out.index + 1
    return out



def compute_shape_stats(frame, target):
    """
    Testa várias formas funcionais entre cada variável explicativa e a inflação.
    Mede o ajustamento dessas formas em amostra através de R².

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame com a variável alvo e as variáveis explicativas.
    target : str
        Nome da variável alvo.

    Returns
    -------
    out : pd.DataFrame
        Tabela com uma linha por variável e as colunas de R² para cada forma
        funcional testada, bem como:
        - `best_model`: forma funcional com maior R²;
        - `best_r2`: valor do melhor R².
    """
    rows = []

    for col in frame.columns:
        if col == target:
            continue

        pair = align_pair(frame, col, target)
        if len(pair) == 0:
            continue

        x = pair['x'].to_numpy()
        y = pair['y'].to_numpy()

        modelos = {}

        X_lin = x.reshape(-1, 1)
        modelos['linear'] = LinearRegression().fit(X_lin, y).score(X_lin, y)

        X_quad = PolynomialFeatures(degree=2, include_bias=False).fit_transform(X_lin)
        modelos['quadratic'] = LinearRegression().fit(X_quad, y).score(X_quad, y)

        X_cubic = PolynomialFeatures(degree=3, include_bias=False).fit_transform(X_lin)
        modelos['cubic'] = LinearRegression().fit(X_cubic, y).score(X_cubic, y)

        if np.all(x > 0):
            X_log = np.log(x).reshape(-1, 1)
            modelos['logarithmic'] = LinearRegression().fit(X_log, y).score(X_log, y)

        if np.all(x >= 0):
            X_sqrt = np.sqrt(x).reshape(-1, 1)
            modelos['sqrt'] = LinearRegression().fit(X_sqrt, y).score(X_sqrt, y)

        if np.all(x != 0):
            X_inv = (1 / x).reshape(-1, 1)
            modelos['inversa'] = LinearRegression().fit(X_inv, y).score(X_inv, y)

        best_model = max(modelos, key=modelos.get)
        best_r2 = modelos[best_model]

        rows.append({
            'variable': col,
            'linear_r2': modelos.get('linear', np.nan),
            'quadratic_r2': modelos.get('quadratic', np.nan),
            'cubic_r2': modelos.get('cubic', np.nan),
            'logarithmic_r2': modelos.get('logarithmic', np.nan),
            'sqrt_r2': modelos.get('sqrt', np.nan),
            'inversa_r2': modelos.get('inversa', np.nan),
            'best_model': best_model,
            'best_r2': best_r2,
        })

    return pd.DataFrame(rows).sort_values('best_r2', ascending=False).reset_index(drop=True)



def compute_explanatory_lags(frame, target, max_lag=12):
    """
    Calcula a correlação entre a inflação no presente e as variáveis explicativas
    com lags de 0 até `max_lag`.

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame com a variável alvo e as variáveis explicativas.
    target : str
        Nome da variável alvo.
    max_lag : int, default=12
        Lag máximo a testar para cada variável explicativa.

    Returns
    -------
    lag_long : pd.DataFrame
        Tabela completa com todas as combinações variável-lag testadas.
    lag_best : pd.DataFrame
        Tabela com a melhor correlação absoluta de Pearson para cada variável
        explicativa, mantendo sempre a inflação no presente.
    """
    rows = []

    for col in frame.columns:
        if col == target:
            continue

        for lag in range(0, max_lag + 1):
            pair = pd.concat(
                [frame[target], frame[col].shift(lag)],
                axis=1,
                keys=['y', 'x']
            ).dropna()

            if len(pair) == 0:
                continue

            pearson = pair['x'].corr(pair['y'], method='pearson')

            rows.append({
                'variable': col,
                'lag': lag,
                'n_obs': len(pair),
                'pearson': pearson,
                'abs_pearson': abs(pearson),
            })

    lag_long = pd.DataFrame(rows)
    lag_best = (
        lag_long
        .sort_values(['variable', 'abs_pearson'], ascending=[True, False])
        .groupby('variable', as_index=False)
        .first()
        .sort_values('abs_pearson', ascending=False)
        .reset_index(drop=True)
    )

    return lag_long, lag_best



def compute_target_autocorrelation(frame, target, max_lag=12):
    """
    Calcula a autocorrelação da variável alvo com os seus próprios lags.

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame que contém a variável alvo.
    target : str
        Nome da variável alvo.
    max_lag : int, default=12
        Lag máximo a testar.

    Returns
    -------
    autocorr_df : pd.DataFrame
        Tabela com uma linha por lag, contendo:
        - `lag`: número de períodos de desfasamento;
        - `n_obs`: número de observações usadas;
        - `pearson`: autocorrelação de Pearson;
        - `abs_pearson`: valor absoluto da autocorrelação.
    """
    rows = []

    for lag in range(1, max_lag + 1):
        pair = pd.concat(
            [frame[target], frame[target].shift(lag)],
            axis=1,
            keys=['y', 'x']
        ).dropna()

        if len(pair) == 0:
            continue

        pearson = pair['x'].corr(pair['y'], method='pearson')

        rows.append({
            'lag': lag,
            'n_obs': len(pair),
            'pearson': pearson,
            'abs_pearson': abs(pearson),
        })

    return pd.DataFrame(rows).sort_values('lag').reset_index(drop=True)



def build_shortlist_table(pairwise_stats, lag_best, shape_stats, top_n=15):
    """
    Constrói a shortlist final usando apenas Pearson em nível como critério de seleção.

    Parameters
    ----------
    pairwise_stats : pd.DataFrame
        Tabela devolvida por `compute_pairwise_stats()`.
    lag_best : pd.DataFrame
        Tabela devolvida por `compute_explanatory_lags()`, contendo o melhor lag
        encontrado para cada variável explicativa.
    shape_stats : pd.DataFrame
        Tabela devolvida por `compute_shape_stats()`.
    top_n : int, default=15
        Número de variáveis a incluir na shortlist final.

    Returns
    -------
    ranking_full : pd.DataFrame
        Ranking completo por `abs_pearson`, enriquecido com informação de lags
        e forma funcional, sem colunas redundantes.
    shortlist : pd.DataFrame
        Tabela final com as `top_n` variáveis selecionadas apenas com base em
        `|Pearson|` em nível.
    """
    ranking_full = (
        pairwise_stats
        .rename(columns={'n_obs': 'n_obs_level'})
        .merge(
            lag_best[
                ['variable', 'lag', 'n_obs', 'pearson', 'abs_pearson']
            ].rename(
                columns={
                    'lag': 'best_lag',
                    'n_obs': 'n_obs_lag',
                    'pearson': 'best_lag_pearson',
                    'abs_pearson': 'best_lag_abs_pearson',
                }
            ),
            on='variable',
            how='left'
        )
        .merge(
            shape_stats[
                [
                    'variable', 'best_model', 'best_r2', 'linear_r2',
                    'quadratic_r2', 'cubic_r2', 'logarithmic_r2',
                    'sqrt_r2', 'inversa_r2'
                ]
            ],
            on='variable',
            how='left'
        )
    )

    shortlist = ranking_full.head(top_n).copy()
    return ranking_full, shortlist



def plot_best_relationship_grid(frame, target, variables, shape_stats, n_cols=3, plot_title=None, x_label=None, y_label=None, bg_color='#ECE6F2'):
    """
    Desenha, para cada variável selecionada, a curva correspondente à forma
    funcional com melhor R² em `shape_stats`.

    Parameters
    ----------
    frame : pd.DataFrame
        DataFrame com a variável alvo e as variáveis explicativas.
    target : str
        Nome da variável alvo.
    variables : list[str]
        Lista de variáveis a representar.
    shape_stats : pd.DataFrame
        Tabela devolvida por `compute_shape_stats()`.
    n_cols : int, default=3
        Número de colunas da grelha de gráficos.

    Returns
    -------
    None
        A função apenas produz os gráficos.
    """
    if not variables:
        print('Não existem variáveis para representar.')
        return

    n_rows = math.ceil(len(variables) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    fig.patch.set_facecolor(bg_color)
    axes = np.atleast_1d(axes).flatten()

    for i, var in enumerate(variables):
        ax = axes[i]
        ax.set_facecolor(bg_color)
        pair = align_pair(frame, var, target)

        if pair.empty:
            ax.text(0.5, 0.5, 'Sem dados válidos', ha='center', va='center')
            ax.set_title(var)
            continue

        x = pair['x'].to_numpy()
        y = pair['y'].to_numpy()
        X = x.reshape(-1, 1)

        row = shape_stats.loc[shape_stats['variable'] == var]
        if row.empty:
            ax.text(0.5, 0.5, 'Sem shape_stats', ha='center', va='center')
            ax.set_title(var)
            continue

        best_model = row.iloc[0]['best_model']
        best_r2 = row.iloc[0]['best_r2']
        x_grid = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)

        if best_model == 'linear':
            model = LinearRegression().fit(X, y)
            y_fit = model.predict(x_grid)
        elif best_model == 'quadratic':
            poly = PolynomialFeatures(degree=2, include_bias=False)
            model = LinearRegression().fit(poly.fit_transform(X), y)
            y_fit = model.predict(poly.transform(x_grid))
        elif best_model == 'cubic':
            poly = PolynomialFeatures(degree=3, include_bias=False)
            model = LinearRegression().fit(poly.fit_transform(X), y)
            y_fit = model.predict(poly.transform(x_grid))
        elif best_model == 'logarithmic':
            model = LinearRegression().fit(np.log(X), y)
            y_fit = model.predict(np.log(x_grid))
        elif best_model == 'sqrt':
            model = LinearRegression().fit(np.sqrt(X), y)
            y_fit = model.predict(np.sqrt(x_grid))
        elif best_model == 'inversa':
            model = LinearRegression().fit(1 / X, y)
            y_fit = model.predict(1 / x_grid)
        else:
            ax.text(0.5, 0.5, 'Modelo não suportado', ha='center', va='center')
            ax.set_title(var)
            continue

        ax.scatter(x, y, s=18, alpha=0.55, color=primary_color)
        ax.plot(x_grid, y_fit, color=secondary_color, linewidth=2)
        title_text = f'{plot_title}\n{best_model} | R²={best_r2:.3f}' if plot_title is not None else f'{var}\nMelhor forma: {best_model} | R²={best_r2:.3f}'
        ax.set_title(title_text)
        ax.set_xlabel(x_label if x_label is not None else var)
        ax.set_ylabel(y_label if y_label is not None else target)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

### **1. Correlação contemporânea em nível**

Nesta etapa avalia-se apenas a associação contemporânea em nível entre cada variável explicativa e a inflação, usando o coeficiente de **Pearson**.

Esta é a única métrica que entra na construção da shortlist final.


In [ ]:
pairwise_stats = compute_pairwise_stats(analysis_df, target)

print(f'Top {top_n} por |Pearson| em nível')
display(
    pairwise_stats.head(top_n)[
        ['variable', 'n_obs', 'pearson', 'abs_pearson', 'rank_pearson_level']
    ]
)

In [ ]:
bg_color = '#ECE6F2'
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)
plot_df = pairwise_stats.head(top_n).sort_values('abs_pearson')
ax.barh(plot_df['variable'], plot_df['abs_pearson'], color=primary_color)
ax.set_title(f'Top {top_n} |Pearson| em nível')
ax.set_xlabel('|Pearson|')
ax.set_ylabel('Variável')
ax.grid(axis='x', linestyle='--', alpha=0.5, color=primary_color)
fig.tight_layout()
plt.show()

### **2. Diagnóstico da forma funcional**

Esta secção não mede correlação no sentido restrito. O objetivo é identificar, para cada variável explicativa, qual a forma funcional que melhor descreve a sua relação com a inflação, com base no ajustamento em amostra medido por `R²`.

São testadas relações lineares, polinomiais e transformações simples, incluindo a forma exponencial.

O resultado é usado apenas como **diagnóstico complementar**, e não como critério de seleção da shortlist.


In [ ]:
shape_stats = compute_shape_stats(analysis_df, target)

print('Top 15 variáveis com melhor ajustamento funcional')
display(
    shape_stats.head(15)[
        [
            'variable', 'best_model', 'best_r2', 'linear_r2', 'quadratic_r2',
            'cubic_r2', 'logarithmic_r2', 'sqrt_r2', 'inversa_r2'
        ]
    ]
)

print('Número de variáveis por melhor forma funcional')
display(
    shape_stats['best_model']
    .value_counts()
    .rename_axis('best_model')
    .reset_index(name='n_variables')
)

### **3. Correlação com lags das variáveis explicativas**

Nesta secção mantém-se a inflação no presente e testam-se lags de `0` até `max_lag` para cada variável explicativa. O objetivo é identificar, para cada variável, o lag que maximiza a correlação absoluta de Pearson com a inflação atual.


In [ ]:
lag_long, lag_best = compute_explanatory_lags(
    analysis_df,
    target=target,
    max_lag=max_lag
)

print(f'Melhor lag por variável explicativa (Top {top_n})')
display(
    lag_best.head(top_n)[
        ['variable', 'lag', 'pearson', 'abs_pearson', 'n_obs']
    ]
)

In [ ]:
lag_best.loc[lag_best['variable'] == 'OILPRICEx_fred-md_Eur']

In [ ]:
top_lag_vars = lag_best.head(6)['variable'].tolist()

n_cols = 3
n_rows = math.ceil(len(top_lag_vars) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

for i, var in enumerate(top_lag_vars):
    ax = axes[i]
    profile = lag_long[lag_long['variable'] == var].sort_values('lag')

    ax.plot(
        profile['lag'],
        profile['pearson'],
        marker='o',
        linestyle='-',
        color=primary_color
    )
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(var)
    ax.set_xlabel('Lag da variável explicativa')
    ax.set_ylabel('Correlação de Pearson')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

**Nota de interpretação**

Para cada variável explicativa, o mais relevante é identificar o lag em que a correlação com a inflação atual atinge maior valor absoluto. Esta secção é apenas complementar e não altera o critério de seleção da shortlist.


### **4. Autocorrelação da inflação**

Aqui analisa-se a correlação da própria inflação com os seus lags, de `1` até `max_lag`, para perceber o grau de persistência temporal da variável alvo.


In [ ]:
target_autocorr = compute_target_autocorrelation(
    analysis_df,
    target=target,
    max_lag=max_lag
)

display(target_autocorr)

In [ ]:
bg_color = "#FFFFFF"
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)
ax.bar(target_autocorr['lag'], target_autocorr['pearson'], color=secondary_color)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Autocorrelação da inflação por lag')
ax.set_xlabel('Lag da inflação')
ax.set_ylabel('Correlação de Pearson')
ax.grid(axis='y', linestyle='--', alpha=0.5, color=secondary_color)
fig.tight_layout()
plt.show()

### **5. Construção da shortlist final**

A shortlist final é construída **apenas** com base no ranking de `|Pearson|` em nível. Os lags e a forma funcional são depois adicionados à tabela apenas para contextualização.


In [ ]:
ranking_full, shortlist = build_shortlist_table(
    pairwise_stats=pairwise_stats,
    lag_best=lag_best,
    shape_stats=shape_stats,
    top_n=top_n
)

print(f'Shortlist final: Top {top_n} por |Pearson| em nível')
display(shortlist)

**Nota de interpretação da tabela**

A shortlist final deve ser lida como um ranking simples das variáveis com maior `|Pearson|` em nível face à inflação. As colunas de lag e de forma funcional servem apenas para complementar a interpretação de cada variável selecionada.


In [ ]:
#shortlist.to_excel('shortlist_variaveis_inflacao_pearson.xlsx', index=False)
#print('Shortlist exportada para shortlist_variaveis_inflacao_pearson.xlsx')

### **6. Visualização detalhada das variáveis finalistas**

Para as variáveis incluídas na shortlist, os gráficos seguintes representam a forma funcional que apresentou o melhor ajustamento em amostra segundo a análise anterior.


In [ ]:
selected_vars = shortlist.head(10)['variable'].tolist()

if selected_vars:
    plot_best_relationship_grid(
        analysis_df,
        target,
        selected_vars,
        shape_stats,
        n_cols=3
    )
else:
    print('A shortlist ficou vazia; não há gráficos de relação para mostrar.')

**Nota de interpretação**

Cada gráfico mostra os pontos observados entre a variável explicativa e a inflação, bem como a curva correspondente à forma funcional que apresentou o melhor ajustamento em amostra. O objetivo é complementar a análise numérica, não redefinir a shortlist.


In [ ]:
# Variáveis com maior best_r2, independentemente da shortlist.
top_r2_vars = shape_stats.sort_values('best_r2', ascending=False).head(1)['variable'].tolist()

display(
    shape_stats.sort_values('best_r2', ascending=False).head(6)[
        ['variable', 'best_model', 'best_r2']
    ]
)


plot_best_relationship_grid(
    analysis_df,
    target,
    top_r2_vars,
    shape_stats,
    n_cols=9,
    x_label='Indice harmonizado de preços ao consumidor: Energia (2010=100)',
    plot_title='Scatter Plot',
    y_label='Inflação (%)',
    bg_color="#FFFFFF"
)


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.pyplot as plt

var = 'PPIPT_ppi'
pair = align_pair(analysis_df, var, target)

x = pair['x'].to_numpy()
y = pair['y'].to_numpy()
x = x.reshape(-1, 1)

model = LinearRegression().fit(x, y)

x_grid = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
y_fit = model.predict(x_grid)

r2_linear = shape_stats.loc[
    shape_stats['variable'] == var, 'linear_r2'
].iloc[0]

pearson_val = pairwise_stats.loc[pairwise_stats['variable'] == var, 'pearson'].iloc[0]

bg_color = "#FFFFFF"
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)

ax.scatter(x, y, s=18, alpha=0.55, color="#7595bc")
ax.plot(x_grid, y_fit, color="#efa374", linewidth=2)

ax.set_title(
    f'Scatter Plot',
    fontname='Times New Roman',
    fontsize=14
)
ax.set_xlabel(
    'PPI (Variação Homóloga, %)',
    fontname='Times New Roman',
    fontsize=12
)
ax.set_ylabel(
    'Taxa de Inflação (%)',
    fontname='Times New Roman',
    fontsize=12
)

ax.tick_params(axis='both', labelsize=12)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontname('Times New Roman')

ax.grid(True, alpha=0.4)

fig.tight_layout()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.show()

Nota: O ppi vs ipc tem um R2 de 0.37 e um rho de 0.6

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import numpy as np
import matplotlib.pyplot as plt

var = 'HICPNG_PT_ea-md'
pair = align_pair(analysis_df, var, target)

x = pair['x'].to_numpy()
y = pair['y'].to_numpy()
X = x.reshape(-1, 1)

poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)

model = LinearRegression().fit(X_poly, y)

x_grid = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
x_grid_poly = poly.transform(x_grid)
y_fit = model.predict(x_grid_poly)

r2_cubic = shape_stats.loc[
    shape_stats['variable'] == var, 'cubic_r2'
].iloc[0]

pearson_val = pairwise_stats.loc[pairwise_stats['variable'] == var, 'pearson'].iloc[0]

bg_color = "#FFFFFF"
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)

ax.scatter(x, y, s=18, alpha=0.55, color=primary_color)
ax.plot(x_grid, y_fit, color=secondary_color, linewidth=2)

ax.set_title(f'Scatter Plot\nCubic | R²={r2_cubic:.3f} | ρ = {pearson_val:.3f}')
ax.set_xlabel('HICP: Bens Não-Energéticos Industriais (2015=100)')
ax.set_ylabel('Inflação (%)')
ax.grid(True, alpha=0.4)

fig.tight_layout()
plt.show()


In [ ]:
pairwise_stats.loc[pairwise_stats['variable'] == 'HICPNG_PT_ea-md']

---

## **Leitura final**

Ao interpretar este notebook, importa separar três planos:

1. **seleção da shortlist**
   - depende apenas de `|Pearson|` em nível;

2. **caracterização temporal**
   - usa os lags das variáveis explicativas e a autocorrelação da inflação como informação complementar;

3. **diagnóstico da forma funcional**
   - usa o `R²` apenas para perceber que tipo de relação melhor ajusta cada variável à inflação.

Se necessário, o passo seguinte poderá passar por uma análise multivariada, já com controlo da colinearidade entre preditores.
